# Process 0 Generator: Autoclaving Event Log

This notebook builds a discrete-event simulation for an autoclaving process and generates an event log in the format expected by the pipeline:

- case_id
- activity
- timestamp_start
- timestamp_end
- object
- object_type
- higher_level_activity
- object_attributes

Activities are logged as <activity>_<object>, for example filling_M1.

Flow per batch:
1. Filling
2. Sterilize on one of 2 machines (prepare -> heat -> hold -> cool -> clean)
3. Packaging on one station

In [ ]:
import simpy
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

RANDOM_SEED = 42
N_BATCHES = 250
MEAN_INTERARRIVAL_MIN = 22.0
SIM_START = pd.Timestamp("2026-01-01 06:00:00")

rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
events = []

def to_timestamp(sim_minutes: float) -> pd.Timestamp:
    return SIM_START + pd.to_timedelta(float(sim_minutes), unit="m")

def sample_duration(activity: str) -> float:
    if activity == "filling":
        return max(1.0, rng.normal(20, 3))
    if activity == "prepare":
        return max(1.0, rng.normal(8, 1.2))
    if activity == "heat":
        return max(1.0, rng.normal(18, 2.5))
    if activity == "hold":
        return max(1.0, rng.normal(24, 3.0))
    if activity == "cool":
        return max(1.0, rng.normal(10, 1.5))
    if activity == "clean":
        return max(1.0, rng.normal(5, 0.7))
    if activity == "packaging":
        return max(1.0, rng.normal(15, 2.0))
    raise ValueError(f"Unknown activity: {activity}")

def log_event(case_id, activity, start_min, end_min, obj, obj_type, higher_level, attrs):
    activity_with_object = f"{activity}_{obj}"
    events.append({
        "case_id": str(case_id),
        "activity": activity_with_object,
        "timestamp_start": to_timestamp(start_min),
        "timestamp_end": to_timestamp(end_min),
        "object": str(obj),
        "object_type": str(obj_type),
        "higher_level_activity": str(higher_level) if higher_level is not None else None,
        "object_attributes": dict(attrs),
    })

def log_machine_event(case_id, activity, start_min, end_min, machine_name, attrs, higher_level=None):
    if higher_level is None:
        higher_level = f"sterilize_{machine_name}" if machine_name in ["M2", "M3"] else f"{activity}_{machine_name}"
    log_event(
        case_id=case_id,
        activity=activity,
        start_min=start_min,
        end_min=end_min,
        obj=machine_name,
        obj_type="machine_state",
        higher_level=higher_level,
        attrs=attrs,
    )

def choose_balanced_machine(resources):
    loads = np.array([res.count + len(res.queue) for res in resources], dtype=float)
    min_load = loads.min()
    candidates = np.where(loads == min_load)[0]
    return int(rng.choice(candidates))

def process_batch(
    env, case_id, batch_size, liquid_density, type_package, variant,
    bottling, autoclaves, packaging_station
    ):
    attrs_base = {
        "batch_size": int(batch_size),
        "liquid_density": float(liquid_density),
        "type_pacakge": str(type_package),
        "type_package": str(type_package),
        "variant": str(variant),
    }

    with bottling.request() as req:
        yield req
        start = env.now
        dur = sample_duration("filling")
        yield env.timeout(dur)
        end = env.now
        log_machine_event(case_id, "filling", start, end, "M1", attrs_base)

    auto_idx = choose_balanced_machine(autoclaves)
    autoclave_name = "M2" if auto_idx == 0 else "M3"
    with autoclaves[auto_idx].request() as req:
        yield req
        attrs_auto = {**attrs_base, "autoclave": autoclave_name}

        for state in ["prepare", "heat", "hold", "cool", "clean"]:
            start = env.now
            dur = sample_duration(state)
            yield env.timeout(dur)
            end = env.now
            log_machine_event(
                case_id,
                state,
                start,
                end,
                autoclave_name,
                attrs_auto,
                higher_level=f"sterilize_{autoclave_name}",
            )

    machine_name = "M4"
    with packaging_station.request() as p_req:
        yield p_req
        attrs_pack = {**attrs_base, "autoclave": autoclave_name, "packaging_machine": machine_name}
        start = env.now
        dur = sample_duration("packaging")
        yield env.timeout(dur)
        end = env.now
        log_machine_event(case_id, "packaging", start, end, machine_name, attrs_pack)

def batch_arrivals(env, n_batches, mean_interarrival, bottling, autoclaves, packaging_station):
    package_types = ["A", "B"]
    package_probs = [0.5, 0.5]

    for i in range(n_batches):
        case_id = f"batch_{i + 1:04d}"
        batch_size = int(max(100, rng.normal(1200, 180)))
        liquid_density = round(float(np.clip(rng.normal(1.03, 0.03), 0.95, 1.12)), 4)
        type_package = str(rng.choice(package_types, p=package_probs))
        variant = "A" if i % 2 == 0 else "B"

        env.process(process_batch(
            env, case_id, batch_size, liquid_density, type_package, variant,
            bottling, autoclaves, packaging_station
        ))
        interarrival = rng.exponential(mean_interarrival)
        yield env.timeout(interarrival)

In [ ]:
env = simpy.Environment()
bottling = simpy.Resource(env, capacity=1)
autoclaves = [simpy.Resource(env, capacity=1) for _ in range(2)]
packaging_station = simpy.Resource(env, capacity=1)

env.process(batch_arrivals(
    env=env,
    n_batches=N_BATCHES,
    mean_interarrival=MEAN_INTERARRIVAL_MIN,
    bottling=bottling,
    autoclaves=autoclaves,
    packaging_station=packaging_station,
))

env.run()

event_log = pd.DataFrame(events)
event_log = event_log.sort_values(["timestamp_start", "case_id"]).reset_index(drop=True)

required_cols = [
    "case_id",
    "activity",
    "timestamp_start",
    "timestamp_end",
    "object",
    "object_type",
    "higher_level_activity",
    "object_attributes",
]
event_log = event_log[required_cols].copy()

production_plan = (
    event_log
    .sort_values(["case_id", "timestamp_start"])
    .groupby("case_id", as_index=False)
    .agg({
        "timestamp_start": "min",
        "object_attributes": "first",
    })
)

print(f"Generated events: {len(event_log):,}")
print(f"Generated cases: {event_log['case_id'].nunique():,}")
print("object_type values:", sorted(event_log["object_type"].unique().tolist()))
print("Activities:", sorted(event_log['activity'].unique().tolist()))
event_log.head(12)

In [ ]:
display(event_log.sample(10, random_state=RANDOM_SEED).sort_values("timestamp_start"))
display(event_log.groupby(["object", "activity"]).size().rename("n_events").reset_index().head(20))
display(production_plan.head())

In [ ]:
repo_root = Path.cwd()
if not (repo_root / "data").exists() and (repo_root.parent / "data").exists():
    repo_root = repo_root.parent

output_dir = (repo_root / "data" / "silver" / "process_0").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

event_log_path = output_dir / "event_log.csv"
production_plan_path = output_dir / "production_plan.csv"

event_log.to_csv(event_log_path, index=False)
production_plan.to_csv(production_plan_path, index=False)

print(f"Saved event_log to: {event_log_path}")
print(f"Saved production_plan to: {production_plan_path}")

In [ ]:
event_log.columns

In [ ]:
#display(event_log.sample(10, random_state=RANDOM_SEED).sort_values("timestamp_start"))
display(event_log.groupby(["object", "activity"]).size().rename("n_events").reset_index().head(20))



display(production_plan.head())

In [ ]:
event_log[['case_id', 'activity', 'timestamp_start', 'timestamp_end', 'object',
       'object_type', 'higher_level_activity', 'object_attributes']]

In [ ]:
attrs_preview = pd.json_normalize(event_log["object_attributes"])

preview_cols = [
    "batch_size",
    "liquid_density",
    "type_pacakge",
]

display(
    pd.concat(
        [
            event_log[["case_id", "activity", "object"]].reset_index(drop=True),
            attrs_preview[preview_cols].reset_index(drop=True),
        ],
        axis=1,
    ).head(40)
 )

In [ ]:
# Normalize object_attributes and prefix every attribute column with "attr_"
attrs_preview = pd.json_normalize(event_log["object_attributes"]).add_prefix("attr_")

# Handle both spellings safely: type_pacakge (current) and type_package (fallback)
if "attr_type_pacakge" not in attrs_preview.columns and "attr_type_package" in attrs_preview.columns:
    attrs_preview["attr_type_pacakge"] = attrs_preview["attr_type_package"]

preview_cols = [
    "attr_batch_size",
    "attr_liquid_density",
    "attr_type_pacakge",
]

result_table = pd.concat(
    [
        event_log[["case_id", "activity", "object", "object_type"]].reset_index(drop=True),
        attrs_preview[preview_cols].reset_index(drop=True),
    ],
    axis=1,
)

# Show the resulting table
display(result_table.head(40))

# Create LaTeX-safe table (escape underscores in headers and string values)
latex_df = result_table.head(15).copy()
latex_df.columns = [c.replace("_", r"\_") for c in latex_df.columns]
for col in latex_df.select_dtypes(include=["object"]).columns:
    latex_df[col] = latex_df[col].str.replace("_", r"\_", regex=False)

# Print LaTeX output
latex_table = latex_df.to_latex(index=False, escape=False)
print(latex_table)

In [ ]:
# Heuristics Miner Petri nets: 1 for material flow + 1 per machine
import pm4py

def mine_and_view_heuristic_net(df, label):
    df_local = df[["case_id", "activity", "timestamp_start"]].copy()
    if df_local.empty:
        print(f"[{label}] skipped: no events")
        return None
    df_local["timestamp_start"] = pd.to_datetime(df_local["timestamp_start"])

    pm_log = pm4py.format_dataframe(
        df_local,
        case_id="case_id",
        activity_key="activity",
        timestamp_key="timestamp_start",
    )
    try:
        net, im, fm = pm4py.discover_petri_net_heuristics(pm_log)
        print(
            f"[{label}] Heuristic net: "
            f"{len(net.places)} places, {len(net.transitions)} transitions, {len(net.arcs)} arcs"
        )
        pm4py.view_petri_net(net, im, fm, format="png")
        return net, im, fm
    except Exception as exc:
        print(f"[{label}] mining failed: {exc}")
        return None

# 1) Material-flow net (single process-level model)
material_log = event_log[event_log["object_type"] == "material_flow"].copy()
_ = mine_and_view_heuristic_net(material_log, "material_flow")

# 2) One machine-state net per machine object
machine_log = event_log[event_log["object_type"] == "machine_state"].copy()
machine_objects = sorted(machine_log["object"].dropna().unique().tolist())
print(f"Machine objects for per-machine nets: {machine_objects}")

for machine_name in machine_objects:
    machine_df = machine_log[machine_log["object"] == machine_name].copy()
    _ = mine_and_view_heuristic_net(machine_df, f"machine::{machine_name}")

In [ ]:
# Directed Follows Graphs (DFG) - no Petri net
import pm4py

def view_dfg_for_subset(df, label):
    if df.empty:
        print(f"[{label}] skipped: no events")
        return
    local = df[["case_id", "activity", "timestamp_start"]].copy()
    local["timestamp_start"] = pd.to_datetime(local["timestamp_start"])

    pm_local = pm4py.format_dataframe(
        local,
        case_id="case_id",
        activity_key="activity",
        timestamp_key="timestamp_start",
    )
    dfg, starts, ends = pm4py.discover_dfg(pm_local)
    print(
        f"[{label}] DFG: {len(dfg)} edges, "
        f"{len(starts)} start activities, {len(ends)} end activities"
    )

    # Some machine logs have only one repeated activity, so DFG has zero edges.
    if len(dfg) == 0:
        print(f"[{label}] no directly-follows edges to visualize (single-activity trace set).")
        return

    pm4py.view_dfg(
        dfg,
        starts,
        ends,
        format="png",
    )

# 1) Process-level DFG (material flow)
material_log = event_log[event_log["object_type"] == "material_flow"].copy()
view_dfg_for_subset(material_log, "material_flow")

# 2) Per-machine DFGs (machine states)
machine_log = event_log[event_log["object_type"] == "machine_state"].copy()
machine_objects = sorted(machine_log["object"].dropna().unique().tolist())
print(f"Machine objects for per-machine DFGs: {machine_objects}")
for machine_name in machine_objects:
    machine_df = machine_log[machine_log["object"] == machine_name].copy()
    view_dfg_for_subset(machine_df, f"machine::{machine_name}")

In [ ]:
material_log.columns

In [ ]:
material_log = event_log[event_log["object_type"] == "material_flow"].copy()

# Expand object_attributes into regular columns to avoid NaN dict rendering
attrs_cols = pd.json_normalize(
    material_log["object_attributes"].apply(lambda x: x if isinstance(x, dict) else {})
).add_prefix("attr_")

# Keep legacy typo and canonical key aligned
if "attr_type_package" not in attrs_cols.columns and "attr_type_pacakge" in attrs_cols.columns:
    attrs_cols["attr_type_package"] = attrs_cols["attr_type_pacakge"]
if "attr_type_pacakge" not in attrs_cols.columns and "attr_type_package" in attrs_cols.columns:
    attrs_cols["attr_type_pacakge"] = attrs_cols["attr_type_package"]

# Timestamps shown only until seconds
result_table = material_log[[
    "case_id", "activity", "timestamp_start", "timestamp_end", "object", "object_type"
 ]].copy()
result_table["timestamp_start"] = pd.to_datetime(result_table["timestamp_start"]).dt.strftime("%Y-%m-%d %H:%M:%S")
result_table["timestamp_end"] = pd.to_datetime(result_table["timestamp_end"]).dt.strftime("%Y-%m-%d %H:%M:%S")

attribute_cols = ["attr_batch_size", "attr_liquid_density", "attr_type_package", "attr_variant"]
for col in attribute_cols:
    if col not in attrs_cols.columns:
        attrs_cols[col] = pd.NA

result_table = pd.concat([
    result_table.reset_index(drop=True),
    attrs_cols[attribute_cols].reset_index(drop=True),
], axis=1)

display(result_table.head(40))

# Create LaTeX-safe table (escape underscores in headers and string values)
latex_df = result_table.head(15).copy()
latex_df.columns = [c.replace("_", r"\_") for c in latex_df.columns]
for col in latex_df.select_dtypes(include=["object"]).columns:
    latex_df[col] = latex_df[col].astype(str).str.replace("_", r"\_", regex=False)

# Print LaTeX output
latex_table = latex_df.to_latex(index=False, escape=False)
print(latex_table)

In [ ]:
material_log = event_log[event_log["object_type"] == "material_flow"].copy()

# Expand object_attributes into regular columns to avoid NaN dict rendering
attrs_cols = pd.json_normalize(
    material_log["object_attributes"].apply(lambda x: x if isinstance(x, dict) else {})
).add_prefix("attr_")

# Keep legacy typo and canonical key aligned
if "attr_type_package" not in attrs_cols.columns and "attr_type_pacakge" in attrs_cols.columns:
    attrs_cols["attr_type_package"] = attrs_cols["attr_type_pacakge"]
if "attr_type_pacakge" not in attrs_cols.columns and "attr_type_package" in attrs_cols.columns:
    attrs_cols["attr_type_pacakge"] = attrs_cols["attr_type_package"]

# Timestamps shown only until seconds
result_table = material_log[[
    "case_id", "activity", "timestamp_start", "timestamp_end", "object", "object_type"
 ]].copy()

result_table["timestamp_start"] = pd.to_datetime(result_table["timestamp_start"]).dt.strftime("%Y-%m-%d %H:%M:%S")
result_table["timestamp_end"] = pd.to_datetime(result_table["timestamp_end"]).dt.strftime("%Y-%m-%d %H:%M:%S")

attribute_cols = ["attr_batch_size", "attr_liquid_density", "attr_type_package", "attr_variant"]
for col in attribute_cols:
    if col not in attrs_cols.columns:
        attrs_cols[col] = pd.NA

result_table = pd.concat([
    result_table.reset_index(drop=True),
    attrs_cols[attribute_cols].reset_index(drop=True),
], axis=1)

display(result_table.head(40))

# Create LaTeX-safe table (escape underscores in headers and string values)
latex_df = result_table.head(15).copy()
latex_df.columns = [c.replace("_", r"\_") for c in latex_df.columns]
for col in latex_df.select_dtypes(include=["object"]).columns:
    latex_df[col] = latex_df[col].astype(str).str.replace("_", r"\_", regex=False)

# Print LaTeX output
latex_table = latex_df.to_latex(index=False, escape=False)
print(latex_table)

In [ ]:
material_log = event_log[event_log["object_type"] == "material_flow"].copy()

# Expand object_attributes into regular columns to avoid NaN dict rendering
attrs_cols = pd.json_normalize(
    material_log["object_attributes"].apply(lambda x: x if isinstance(x, dict) else {})
).add_prefix("attr_")

# Keep legacy typo and canonical key aligned
if "attr_type_package" not in attrs_cols.columns and "attr_type_pacakge" in attrs_cols.columns:
    attrs_cols["attr_type_package"] = attrs_cols["attr_type_pacakge"]
if "attr_type_pacakge" not in attrs_cols.columns and "attr_type_package" in attrs_cols.columns:
    attrs_cols["attr_type_pacakge"] = attrs_cols["attr_type_package"]

# Timestamps shown only until seconds
result_table = material_log[[
    "case_id", "activity", "timestamp_start", "timestamp_end"
 ]].copy()

result_table["timestamp_start"] = pd.to_datetime(result_table["timestamp_start"]).dt.strftime("%Y-%m-%d %H:%M:%S")
result_table["timestamp_end"] = pd.to_datetime(result_table["timestamp_end"]).dt.strftime("%Y-%m-%d %H:%M:%S")

attribute_cols = ["attr_variant"]
for col in attribute_cols:
    if col not in attrs_cols.columns:
        attrs_cols[col] = pd.NA

result_table = pd.concat([
    result_table.reset_index(drop=True),
    attrs_cols[attribute_cols].reset_index(drop=True),
], axis=1)

display(result_table.head(40))

# Create LaTeX-safe table (escape underscores in headers and string values)
latex_df = result_table.head(15).copy()
latex_df.columns = [c.replace("_", r"\_") for c in latex_df.columns]
for col in latex_df.select_dtypes(include=["object"]).columns:
    latex_df[col] = latex_df[col].astype(str).str.replace("_", r"\_", regex=False)

# Print LaTeX output
latex_table = latex_df.to_latex(index=False, escape=False)
print(latex_table)

In [ ]:
machine_log = event_log[event_log["object_type"] == "machine_state"].copy()

# Expand object_attributes into columns
attrs_cols = pd.json_normalize(
    machine_log["object_attributes"].apply(lambda x: x if isinstance(x, dict) else {})
).add_prefix("attr_")

# Ensure attribute column exists
if "attr_variant" not in attrs_cols.columns:
    attrs_cols["attr_variant"] = pd.NA

# Build result table and show higher-level activity as activity
result_table = machine_log[["case_id", "higher_level_activity", "timestamp_start"]].copy()
result_table = result_table.rename(columns={"higher_level_activity": "activity"})

# Format timestamp
result_table["timestamp_start"] = pd.to_datetime(
    result_table["timestamp_start"]
).dt.strftime("%Y-%m-%d %H:%M")

# Keep only last 3 digits of case_id
result_table["case_id"] = result_table["case_id"].astype(str).str[-3:]

# Add attribute column
result_table["attribute"] = attrs_cols["attr_variant"].values

display(result_table.head(40))

# LaTeX export
latex_df = result_table.head(15).copy()
latex_df.columns = [c.replace("_", r"\_") for c in latex_df.columns]

for col in latex_df.select_dtypes(include=["object"]).columns:
    latex_df[col] = latex_df[col].astype(str).str.replace("_", r"\_", regex=False)

latex_table = latex_df.to_latex(index=False, escape=False)
print(latex_table)

In [ ]:
event_log["object"].value_counts()

In [ ]:
machine_log = event_log[
    (event_log["object_type"] == "machine_state") &
    (event_log["object"] == "M2")
].copy()

# Expand object_attributes into columns
attrs_cols = pd.json_normalize(
    machine_log["object_attributes"].apply(lambda x: x if isinstance(x, dict) else {})
).add_prefix("attr_")

# Ensure attribute column exists
if "attr_variant" not in attrs_cols.columns:
    attrs_cols["attr_variant"] = pd.NA

# Build result table with required columns
result_table = machine_log[["case_id", "activity", "timestamp_start"]].copy()

# Format timestamp
result_table["timestamp_start"] = pd.to_datetime(
    result_table["timestamp_start"]
).dt.strftime("%Y-%m-%d %H:%M")

# Keep only last 3 digits of case_id
result_table["case_id"] = result_table["case_id"].astype(str).str[-3:]

# Add attribute column
result_table["attribute"] = attrs_cols["attr_variant"].values

display(result_table.head(40))

# LaTeX export
latex_df = result_table.head(15).copy()
latex_df.columns = [c.replace("_", r"\_") for c in latex_df.columns]

for col in latex_df.select_dtypes(include=["object"]).columns:
    latex_df[col] = latex_df[col].astype(str).str.replace("_", r"\_", regex=False)

latex_table = latex_df.to_latex(index=False, escape=False)
print(latex_table)

In [ ]:
machine_log = event_log[
    (event_log["object_type"] == "machine_state") 
    #(event_log["object"] == "M2")
].copy()

# Expand object_attributes into columns
attrs_cols = pd.json_normalize(
    machine_log["object_attributes"].apply(lambda x: x if isinstance(x, dict) else {})
).add_prefix("attr_")

# Ensure attribute column exists
if "attr_variant" not in attrs_cols.columns:
    attrs_cols["attr_variant"] = pd.NA

# Build result table with required columns
result_table = machine_log[["case_id", "activity", "timestamp_start"]].copy()

# Format timestamp
result_table["timestamp_start"] = pd.to_datetime(
    result_table["timestamp_start"]
).dt.strftime("%Y-%m-%d %H:%M")

# Keep only last 3 digits of case_id
result_table["case_id"] = result_table["case_id"].astype(str).str[-3:]

# Add attribute column
result_table["attribute"] = attrs_cols["attr_variant"].values

display(result_table.head(40))

# LaTeX export
latex_df = result_table.head(15).copy()
latex_df.columns = [c.replace("_", r"\_") for c in latex_df.columns]

for col in latex_df.select_dtypes(include=["object"]).columns:
    latex_df[col] = latex_df[col].astype(str).str.replace("_", r"\_", regex=False)

latex_table = latex_df.to_latex(index=False, escape=False)
print(latex_table)

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------------
# 1) Pick at least 2 activities
# -----------------------------------
machine_log = event_log[event_log["object_type"] == "machine_state"].copy()
activities = machine_log["activity"].dropna().astype(str).unique().tolist()
chosen_activities = activities[:2]

print("Chosen activities:", chosen_activities)

# -----------------------------------
# 2) Create synthetic varying curves
# -----------------------------------
rows = []
start_time = pd.Timestamp("2026-01-01 08:00:00")

for i, act in enumerate(chosen_activities):
    n_seconds = 60

    timestamps = pd.date_range(
        start=start_time + pd.Timedelta(minutes=2*i),
        periods=n_seconds,
        freq="1s"
    )

    t = np.arange(n_seconds)

    # Values only in the hundreds, with clear variation
    if i == 0:
        values = (
            180
            + 35 * np.sin(t / 6)
            + 18 * np.cos(t / 11)
            + np.linspace(0, 40, n_seconds)
            + np.random.normal(0, 4, n_seconds)
        )
    else:
        values = (
            140
            + 28 * np.sin(t / 4)
            + 12 * np.cos(t / 9)
            + 30 * np.exp(-((t - 30) ** 2) / 180)
            + np.random.normal(0, 3, n_seconds)
        )

    # keep values positive and in a nice range
    values = np.clip(values, 100, 299.99)

    tmp = pd.DataFrame({
        "timestamp": timestamps,
        "value": np.round(values, 2),
        "activity": "",
        "case_id": ""
    })

    rows.append(tmp)

energy_curve_table = pd.concat(rows, ignore_index=True)

display(energy_curve_table.head(80))

# -----------------------------------
# 3) Export to LaTeX
# -----------------------------------
latex_df = energy_curve_table.head(30).copy()
latex_df["timestamp"] = latex_df["timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S")
latex_df["value"] = latex_df["value"].map(lambda x: f"{x:.2f}")
latex_df.columns = [c.replace("_", r"\_") for c in latex_df.columns]

for col in latex_df.select_dtypes(include=["object"]).columns:
    latex_df[col] = latex_df[col].astype(str).str.replace("_", r"\_", regex=False)

latex_table = latex_df.to_latex(index=False, escape=False)
print(latex_table)

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------------
# 1) Pick at least 2 activities
# -----------------------------------
machine_log = event_log[event_log["object_type"] == "machine_state"].copy()
activities = machine_log["activity"].dropna().astype(str).unique().tolist()
chosen_activities = activities[:2]
print("Chosen activities:", chosen_activities)

# -----------------------------------
# 2) Create synthetic varying curves
# -----------------------------------
rows = []
start_time = pd.Timestamp("2026-01-01 08:00:00")
case_id = "001"   # one numeric case id with leading zeros

for i, act in enumerate(chosen_activities):
    n_seconds = 60
    timestamps = pd.date_range(
        start=start_time + pd.Timedelta(minutes=2*i),
        periods=n_seconds,
        freq="1s"
    )
    t = np.arange(n_seconds)

    if i == 0:
        values = (
            180
            + 35 * np.sin(t / 6)
            + 18 * np.cos(t / 11)
            + np.linspace(0, 40, n_seconds)
            + np.random.normal(0, 4, n_seconds)
        )
    else:
        values = (
            140
            + 28 * np.sin(t / 4)
            + 12 * np.cos(t / 9)
            + 30 * np.exp(-((t - 30) ** 2) / 180)
            + np.random.normal(0, 3, n_seconds)
        )

    values = np.clip(values, 100, 299.99)

    tmp = pd.DataFrame({
        "value": np.round(values, 2),
        "timestamp": timestamps,
        "activity": act,
        "case_id": case_id
    })
    rows.append(tmp)

energy_curve_table = pd.concat(rows, ignore_index=True)

# -----------------------------------
# 3) Export to LaTeX
#    Take rows from BOTH activities
# -----------------------------------
latex_df = pd.concat([
    energy_curve_table[energy_curve_table["activity"] == chosen_activities[0]].head(15),
    energy_curve_table[energy_curve_table["activity"] == chosen_activities[1]].head(15)
], ignore_index=True)

# Reset timestamps so they are continuous in the printed LaTeX table
latex_start_time = pd.Timestamp("2026-01-01 08:00:00")
latex_df["timestamp"] = pd.date_range(
    start=latex_start_time,
    periods=len(latex_df),
    freq="1s"
)

latex_df["timestamp"] = latex_df["timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S")
latex_df["value"] = latex_df["value"].map(lambda x: f"{x:.2f}")

latex_df.columns = [c.replace("_", r"\_") for c in latex_df.columns]

for col in latex_df.select_dtypes(include=["object"]).columns:
    latex_df[col] = latex_df[col].astype(str).str.replace("_", r"\_", regex=False)

latex_table = latex_df.to_latex(index=False, escape=False)
print(latex_table)